# Content That Converts — V2 Analysis
### Groupon Case Study: End-to-End AI Content Optimization Pipeline

**Author:** Leo B | **Model:** llama-3.3-70b-versatile via Groq | **Dataset:** 500 deals

---
This notebook walks through:
1. Dataset overview & feature distributions
2. Statistical correlation analysis (Pearson + Spearman + RF feature importance)
3. Top vs bottom performer comparison
4. Category-stratified CVR analysis
5. Content quality scorer validation
6. Pipeline results: before/after score comparison
7. LLM judge breakdown per deal
8. Sample before/after showcase
9. Full evaluation summary table

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

# Styling
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
PALETTE = sns.color_palette("rocket_r", 8)
GREEN, RED, BLUE = '#2ecc71', '#e74c3c', '#3498db'

print("✓ Imports OK")

✓ Imports OK


## 1. Dataset Overview

In [ ]:
# Load data
df = pd.read_csv('../data/deals.csv')
analyzer = SentimentIntensityAnalyzer()

# Extract NLP Features
df['desc_length'] = df['description'].fillna('').apply(len)
df['readability'] = df['description'].fillna('').apply(lambda x: flesch_kincaid_grade(x) if x else 0.0)
df['sentiment'] = df['description'].fillna('').apply(lambda x: analyzer.polarity_scores(x)['compound'] if x else 0.0)
df['urgency_mentions'] = df['description'].str.lower().str.count('hurry|limited|today|now')

print(f"Data shape: {df.shape}")
df.head()

Data shape: (500, 38)


,deal_id,category,subcategory,geo,title,description,fine_print,num_options,option_names,price,...,weekly_orders_w7,weekly_udvs_w8,weekly_orders_w8,cvr,aov,refund_rate,desc_length,readability,sentiment,urgency_mentions
0,d04a54f4-c8aa-4083-bba5-9b67df39c6e8,Home Services,HVAC,Seattle,Termite Inspection & Treatment Plan at GreenLe...,"What We Offer: Our licensed, bonded, and insur...",Not valid on holidays; Promotional value expir...,4,"Single Room, Multi-Room Package, Complete Home...",105,...,134,1793,148,0.079365,88.62,0.0793,938,10.050483,0.8269,1
1,d94bb39f-843c-4ac6-9a93-7dbabd1314ed,Health & Fitness,Chiropractic,Chicago,Phentermine Weight-Loss Program w/ Consultatio...,What We Offer: This package includes an initia...,48-hour cancellation policy; Photo ID required...,3,"Option 1, Option 2, Option 3",56,...,30,1224,48,0.047453,54.59,0.0784,886,11.712917,0.8998,1
2,e2c08629-c85e-4e6e-8bce-1c9b47008c5b,Health & Fitness,Weight Loss,Chicago,Restore & Recharge at Vitality Dental,An opportunity to enjoy premium service at a f...,Must redeem within 30 days; Expires 90 days fr...,4,"1 Visit, 3 Visits, 5 Visits, 10 Visits",35,...,29,814,27,0.037258,47.54,0.1044,211,7.713929,0.9246,0
3,7c765ecb-e12b-4d43-ad86-f249a9ca99ba,Beauty & Spas,Hair,Los Angeles,Revitalize Your Look w/ Amazing Services at Sa...,Experience the best that Los Angeles has to of...,No refunds after redemption; Valid at Los Ange...,4,"Mini Refresh, Classic Treatment, Luxury Packag...",48,...,66,2985,64,0.023428,65.54,0.0943,227,7.152635,0.9319,1
4,872bbb9d-a910-415a-b804-cf2316c0496c,Automotive,Inspection,Los Angeles,Amazing Auto Service at MasterMech Tire & Auto,Save on this top-rated experience in Los Angel...,Subject to availability; Merchant is solely re...,1,1 Session,61,...,17,1185,14,0.012918,74.76,0.0542,196,5.598750,0.7506,0


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CVR Distribution
axes[0].hist(df['conversion_rate'] * 100, bins=40, color=BLUE, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Conversion Rate (%)')
axes[0].set_title('CVR Distribution')
axes[0].axvline(df['conversion_rate'].mean() * 100, color=RED, ls='--', label=f"Mean {df['conversion_rate'].mean()*100:.1f}%")
axes[0].legend(fontsize=9)

# Category CVR
cat_cvr = df.groupby('category')['conversion_rate'].mean().sort_values()
axes[1].barh(cat_cvr.index, cat_cvr.values * 100, color=PALETTE[:len(cat_cvr)])
axes[1].set_xlabel('Mean CVR (%)')
axes[1].set_title('Mean CVR by Category')
for i, v in enumerate(cat_cvr.values):
    axes[1].text(v * 100 + 0.05, i, f'{v*100:.1f}%', va='center', fontsize=9)

# Geo CVR
geo_cvr = df.groupby('geo')['conversion_rate'].mean().sort_values().tail(10)
axes[2].barh(geo_cvr.index, geo_cvr.values * 100, color=PALETTE[:len(geo_cvr)])
axes[2].set_xlabel('Mean CVR (%)')
axes[2].set_title('Mean CVR by Geo (Top 10)')

plt.suptitle('Dataset Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../docs/fig_dataset_overview.png', bbox_inches='tight')
plt.show()

## 2. Feature Correlation Analysis

In [ ]:
with open('../docs/analysis_findings.json') as f:
    findings = json.load(f)

corr_data = findings['pearson_correlations']
features_sorted = sorted(corr_data.items(), key=lambda x: abs(x[1]['r']), reverse=True)[:16]

labels = [k.replace('_', ' ') for k, _ in features_sorted]
rs = [v['r'] for _, v in features_sorted]
ps = [v['p'] for _, v in features_sorted]
colors = [GREEN if r > 0 else RED for r in rs]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(labels[::-1], rs[::-1], color=colors[::-1], edgecolor='white', linewidth=0.3)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with Conversion Rate')
ax.set_title('Feature Correlations with CVR\n(all shown are p < 0.05)', fontsize=13)

for i, (bar, r, p) in enumerate(zip(bars, rs[::-1], ps[::-1])):
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else '*')
    ax.text(r + (0.005 if r >= 0 else -0.005),
            bar.get_y() + bar.get_height()/2,
            f'{r:+.3f} {sig}', va='center', ha='left' if r >= 0 else 'right', fontsize=8.5)

plt.tight_layout()
plt.savefig('../docs/fig_correlations.png', bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest Feature Importances
rf_importances = findings['rf_feature_importances']
rf_sorted = sorted(rf_importances.items(), key=lambda x: x[1], reverse=True)[:12]
rf_labels = [k.replace('_', ' ') for k, _ in rf_sorted]
rf_vals = [v for _, v in rf_sorted]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(rf_labels[::-1], rf_vals[::-1], color=BLUE, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Mean Decrease in Impurity (Normalized)')
ax.set_title(f'Random Forest Feature Importances\n(200 trees, 5-fold CV R²={findings["cv_r2"]:.3f})', fontsize=13)
for bar, val in zip(bars, rf_vals[::-1]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../docs/fig_rf_importances.png', bbox_inches='tight')
plt.show()

## 3. Top vs Bottom Performer Comparison

In [ ]:
top_bottom = findings['top_bottom_comparison']
comp_features = ['desc_word_count', 'specificity_count', 'structure_section_count',
                 'social_proof_count', 'title_length', 'image_quality_score',
                 'desc_flesch_ease', 'fine_print_restriction_count']

top_vals = [top_bottom['top']['means'].get(f, 0) for f in comp_features]
bot_vals = [top_bottom['bottom']['means'].get(f, 0) for f in comp_features]
labels_tb = [f.replace('_', ' ') for f in comp_features]

x = np.arange(len(comp_features))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - width/2, top_vals, width, label='Top 20% CVR', color=GREEN, edgecolor='white')
b2 = ax.bar(x + width/2, bot_vals, width, label='Bottom 20% CVR', color=RED, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(labels_tb, rotation=35, ha='right', fontsize=9)
ax.legend()
ax.set_title('Top vs Bottom Performer Feature Comparison\n(Mean values, normalized per feature)', fontsize=13)
ax.set_ylabel('Mean Value')
plt.tight_layout()
plt.savefig('../docs/fig_top_bottom.png', bbox_inches='tight')
plt.show()

print(f"Top 20% mean CVR:    {top_bottom['top']['mean_cvr']:.4f} ({top_bottom['top']['count']} deals)")
print(f"Bottom 20% mean CVR: {top_bottom['bottom']['mean_cvr']:.4f} ({top_bottom['bottom']['count']} deals)")

## 4. Category-Stratified CVR Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot of CVR by category
cat_order = df.groupby('category')['conversion_rate'].median().sort_values().index
df_plot = df.copy()
df_plot['cvr_pct'] = df_plot['conversion_rate'] * 100
sns.boxplot(data=df_plot, y='category', x='cvr_pct', order=cat_order,
            palette='rocket_r', ax=axes[0], linewidth=0.8)
axes[0].set_xlabel('Conversion Rate (%)')
axes[0].set_ylabel('')
axes[0].set_title('CVR Distribution by Category')

# Scatter: desc_word_count vs CVR, colored by category
cats = df['category'].unique()
cat_colors = dict(zip(cats, sns.color_palette('hls', len(cats))))
for cat in cats:
    sub = df[df['category'] == cat]
    axes[1].scatter(sub['desc_word_count'], sub['conversion_rate']*100,
                    label=cat, alpha=0.5, s=18, color=cat_colors[cat])
axes[1].set_xlabel('Description Word Count')
axes[1].set_ylabel('CVR (%)')
axes[1].set_title(f'Word Count vs CVR (r={findings["pearson_correlations"]["desc_word_count"]["r"]:+.3f})')
axes[1].legend(fontsize=7, ncol=2)

# Add trend line
m, b, r, p, _ = stats.linregress(df['desc_word_count'], df['conversion_rate']*100)
x_line = np.linspace(df['desc_word_count'].min(), df['desc_word_count'].max(), 100)
axes[1].plot(x_line, m*x_line + b, color='black', lw=1.5, ls='--', alpha=0.7)

plt.tight_layout()
plt.savefig('../docs/fig_category_analysis.png', bbox_inches='tight')
plt.show()

## 5. Pipeline Results: Before/After Score Comparison

In [ ]:
results_df = pd.read_csv('../results/latest_results.csv')
print(f"Pipeline processed {len(results_df)} deals")
print(f"Verdicts: {results_df['verdict'].value_counts().to_dict()}")
print(f"Mean composite score delta: +{results_df['composite_delta'].mean():.1f} pts")
print(f"Mean LLM judge score delta: +{results_df['judge_score_delta'].mean():.1f} pts")

results_df[['deal_id','category','cvr_original','composite_orig','composite_new','composite_delta','verdict']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Before/After composite scores
x = np.arange(len(results_df))
width = 0.38
labels_short = [f"#{i+1} {r['category'][:8]}" for i, r in results_df.iterrows()]

axes[0].bar(x - width/2, results_df['composite_orig'], width, label='Before', color=RED, alpha=0.85)
axes[0].bar(x + width/2, results_df['composite_new'], width, label='After', color=GREEN, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_short, rotation=40, ha='right', fontsize=8)
axes[0].set_ylabel('Composite Content Score (0–100)')
axes[0].set_title('Before vs After: Composite Content Score')
axes[0].legend()
axes[0].set_ylim(0, 100)

# LLM judge scores
axes[1].bar(x - width/2, results_df['judge_orig_score'], width, label='Orig (LLM judge)', color=RED, alpha=0.85)
axes[1].bar(x + width/2, results_df['judge_new_score'], width, label='Rewritten (LLM judge)', color=GREEN, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_short, rotation=40, ha='right', fontsize=8)
axes[1].set_ylabel('Blinded A/B LLM Judge Score (0–100)')
axes[1].set_title('Before vs After: Blinded LLM Judge Score')
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.suptitle(f'Pipeline Results — 10 Lowest-CVR Deals Rewritten | Mean score delta: +{results_df["composite_delta"].mean():.1f} pts',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../docs/fig_before_after.png', bbox_inches='tight')
plt.show()

## 6. LLM Judge Multi-Axis Breakdown

In [ ]:
with open('../results/latest_results.json') as f:
    results_json = json.load(f)

# Extract per-axis judge scores for all deals
axes_names = ['clarity', 'persuasiveness', 'specificity', 'accuracy', 'professionalism']
orig_axes_mean = {a: 0 for a in axes_names}
new_axes_mean = {a: 0 for a in axes_names}
n = 0

for r in results_json:
    ev = json.loads(r['full_evaluation'])
    judge = ev.get('llm_judge', {})
    bd_new = judge.get('breakdown_new', {})
    bd_orig = judge.get('breakdown_orig', {})
    if bd_new and bd_orig:
        for a in axes_names:
            orig_axes_mean[a] += bd_orig.get(a, 0)
            new_axes_mean[a] += bd_new.get(a, 0)
        n += 1

for a in axes_names:
    orig_axes_mean[a] /= n
    new_axes_mean[a] /= n

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(axes_names))
width = 0.38
ax.bar(x - width/2, [orig_axes_mean[a] for a in axes_names], width, label='Original', color=RED, alpha=0.85)
ax.bar(x + width/2, [new_axes_mean[a] for a in axes_names], width, label='Rewritten', color=GREEN, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([a.title() for a in axes_names])
ax.set_ylabel('Mean Score (0–20 per axis)')
ax.set_title('LLM Judge: Mean Score by Axis (Blinded A/B, 5 axes × 20 pts each)', fontsize=12)
ax.legend()
ax.set_ylim(0, 20)
for i, a in enumerate(axes_names):
    ax.text(i - width/2, orig_axes_mean[a] + 0.3, f'{orig_axes_mean[a]:.1f}', ha='center', fontsize=8.5)
    ax.text(i + width/2, new_axes_mean[a] + 0.3, f'{new_axes_mean[a]:.1f}', ha='center', fontsize=8.5)
plt.tight_layout()
plt.savefig('../docs/fig_judge_axes.png', bbox_inches='tight')
plt.show()

## 7. Sample Before/After Deal Showcase

In [ ]:
from IPython.display import display, HTML

for r in results_json[:2]:
    ev = json.loads(r['full_evaluation'])
    judge = ev.get('llm_judge', {})
    html = f"""
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0; font-family:sans-serif; max-width:900px">
      <h3 style="margin:0 0 4px 0; color:#2c3e50">Deal #{results_json.index(r)+1}: {r['merchant_name']} — {r['category']} ({r['geo']})</h3>
      <p style="margin:0; color:#888; font-size:12px">CVR: {r['cvr_original']*100:.2f}% | Score: {r['composite_orig']:.0f} → {r['composite_new']:.0f} (+{r['composite_delta']:.0f} pts) | Verdict: <b style="color:{'green' if r['verdict']=='PASS' else 'orange'}">{r['verdict']}</b></p>
      <table style="width:100%; margin-top:10px; border-collapse:collapse">
        <tr>
          <td style="width:50%; vertical-align:top; padding-right:10px">
            <b style="color:{RED}">ORIGINAL</b>
            <p><b>Title:</b> {r['original_title']}</p>
            <p><b>Description:</b> {r['original_desc']}</p>
            <p style="color:#aaa; font-size:11px">LLM Judge: {judge.get('orig_score', '?')}/100</p>
          </td>
          <td style="width:50%; vertical-align:top; padding-left:10px; border-left:2px solid #eee">
            <b style="color:{GREEN}">REWRITTEN</b>
            <p><b>Title:</b> {r['improved_title']}</p>
            <p><b>Description:</b> {r['improved_desc']}</p>
            <p style="color:#555; font-size:11px">LLM Judge: {judge.get('new_score', '?')}/100 | ROUGE-L: {r['rouge_l']:.3f}</p>
          </td>
        </tr>
      </table>
      <p style="background:#f9f9f9; padding:8px; border-radius:4px; font-size:11px; color:#555; margin-top:8px">
        <b>AI Reasoning:</b> {r['reasoning']}
      </p>
    </div>
    """
    display(HTML(html))

## 8. Full Evaluation Summary Table

In [ ]:
summary_cols = [
    'rank', 'category', 'geo', 'cvr_original',
    'composite_orig', 'composite_new', 'composite_delta',
    'words_added', 'specificity_delta', 'rouge_l',
    'judge_orig_score', 'judge_new_score', 'judge_score_delta', 'verdict'
]

display_df = results_df[summary_cols].copy()
display_df['cvr_original'] = (display_df['cvr_original'] * 100).round(2).astype(str) + '%'
display_df.columns = ['Rank','Category','Geo','CVR','Score Before','Score After','Score Δ',
                       'Words Added','Spec Δ','ROUGE-L','Judge Before','Judge After','Judge Δ','Verdict']

def color_verdict(val):
    color = 'green' if val == 'PASS' else ('orange' if val == 'MARGINAL' else 'red')
    return f'color: {color}; font-weight: bold'

display_df.style.applymap(color_verdict, subset=['Verdict']) \
    .background_gradient(subset=['Score Δ', 'Judge Δ'], cmap='Greens') \
    .format({'ROUGE-L': '{:.3f}'}) \
    .set_caption(f'Pipeline Results — {len(display_df)} Deals | Mean Score Δ: +{results_df["composite_delta"].mean():.1f} | Mean Judge Δ: +{results_df["judge_score_delta"].mean():.1f}')